# NLP Assignment 2: Apply Word2Vec to Downstream Tasks

In this assignment, you will perform two tasks using Word2Vec:
- Unsupervised PoS tagging using K-Means
- Unsupervised Extractive Summarisation

**We will randomly run 15 submissions. If any bug occurs during execution OR the results fall outside a reasonable reproducible range compared to the logged results, the submission will receive 0 mark for this assignment.**

## 0. Load pre-trained Glove embeddings

In this step, we load the pre-trained GloVe word embeddings from the text file specified by glove_path.
The function load_glove reads the file line by line and processes each entry by:
- Splitting each line into the token (word) and its corresponding embedding values;
- Converting the embedding values from strings to float32 arrays;
- Cleaning the token by removing non-breaking spaces and skipping tokens that contain whitespace or are empty;
- Optionally limiting the number of loaded lines through the max_lines parameter.

The final result is stored in the dictionary glove_vectors. Please download the model (glove.2024.wikigiga.100d.zip, 560MB) at https://nlp.stanford.edu/projects/glove/.

In [ ]:
import numpy as np
from typing import Optional

def load_glove(path: str, dim: int = 100, max_lines: Optional[int] = None) -> dict:
    vectors = {}
    nb_space = u"\xa0"
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_lines is not None and i >= max_lines:
                break
            parts = line.strip().split()
            if len(parts) <= dim:
                continue
            tail = parts[-dim:]
            try:
                vec = np.array([float(x) for x in tail], dtype=np.float32)
            except ValueError:
                continue
            token = " ".join(parts[:-dim]).replace(nb_space, " ").strip()
            if " " in token or token == "":
                continue  
            vectors[token] = vec
    return vectors

glove_path = "path_to_your_file" # Replace path_to_your_file with the path to the glove embedding.
glove_vectors = load_glove(glove_path)

print(f"GloVe Loaded. Vocab size: {len(glove_vectors):,}")

## 1. Apply Word2Vec to Unsupervised PoS Tagging.
In this step, we use word embeddings to perform unsupervised part-of-speech (PoS) tagging through clustering.
The key idea is that words with similar syntactic functions tend to appear in similar contexts and therefore have similar vector representations in the embedding space.

The process involves:
- Using pre-trained Word2Vec (or GloVe) embeddings to represent each word as a dense vector;
- Grouping words into clusters based on their embedding similarity using an unsupervised clustering algorithm (e.g. K-Means);
- Evaluating the quality of the induced clusters against gold PoS tags using information-theoretic metrics.
- Visualise the clustered embeddings.

In this task, we will use K-Means to group text from brown corpus.

### 1.1 Download the Brown Corpus.
We download the brown corpus from nltk to obtain the sentences and corresponding PoS tags. We use the first 500 instances for this task.

The number of unique PoS tags in this subset will be used as the number of clusters (k) in the later clustering step.

In [ ]:
import nltk

nltk.download('brown')
nltk.download('universal_tagset')

from nltk.corpus import brown

tagged_sents = brown.tagged_sents(tagset='universal')[:500]
upos_tags = [t for sent in tagged_sents for (_, t) in sent]


# Calculate the number of PoS tags, which will be used as the k for clustering.
unique_upos = sorted(set(upos_tags))
print(f"PoS tag num: {len(unique_upos)}")
print("PoS list:", unique_upos)


### 1.2 Build Token Embeddings and Gold Tags

In this step, we align the words in the Brown corpus with the pre-trained GloVe embeddings and extract the corresponding PoS tags.

The process involves:
- Collecting words from the tagged sentences and converting them to lowercase;
- Checking whether each word exists in the GloVe vocabulary;
-- If in-vocabulary, we store its embedding and gold PoS tag;
-- If out-of-vocabulary (OOV), we skip it;

The outputs include:
- token_embeddings: mapping from each unique in-vocabulary word to its embedding vector;
- vocab_words: list of unique in-vocabulary word types;
- input_tokens and gold_tags: word-level tokens and their corresponding PoS tags (for evaluation only);
- X contains the GloVe embeddings of unique word types with the shape (V, D), where V is the number of unique in-vocab word types and D is the embedding dimension.


In [ ]:
import numpy as np
from typing import Dict, List, Tuple

def build_token_embeddings_and_gold_tags(
    tagged_sents: List[List[Tuple[str, str]]],
    glove_vectors: Dict[str, np.ndarray],
    lowercase: bool = True
):
    token_embeddings: Dict[str, np.ndarray] = {}
    input_tokens: List[str] = []
    gold_tags: List[str] = []

    total_tokens = 0
    iv_count = 0
    oov_count = 0

    for sent in tagged_sents:
        for w, tag in sent:
            total_tokens += 1
            w_proc = w.lower() if lowercase else w

            if w_proc in glove_vectors:
                iv_count += 1
                input_tokens.append(w_proc)
                gold_tags.append(tag)

                if w_proc not in token_embeddings:
                    token_embeddings[w_proc] = glove_vectors[w_proc]
            else:
                oov_count += 1

    vocab_words = list(token_embeddings.keys())
    if len(vocab_words) == 0:
        raise ValueError("Please check your data.")
    d = token_embeddings[vocab_words[0]].shape[0]
    X = np.vstack([token_embeddings[w] for w in vocab_words]).astype(np.float32)
    assert X.shape == (len(vocab_words), d)

    stats = {
        "total_tokens": total_tokens,
        "in_vocab_tokens": iv_count,
        "oov_tokens": oov_count,
        "oov_ratio": oov_count / max(total_tokens, 1),
        "vocab_size_unique_in_glove": len(vocab_words),
        "embedding_dim": d,
    }
    return token_embeddings, vocab_words, X, input_tokens, gold_tags, stats


token_embeddings, vocab_words, X, input_tokens, gold_tags, stats = build_token_embeddings_and_gold_tags(
    tagged_sents=tagged_sents,
    glove_vectors=glove_vectors,
    lowercase=True
)


print("Finish building")
print(f"- Total token num: {stats['total_tokens']}")
print(f"- In-vocab token num: {stats['in_vocab_tokens']}")
print(f"- OOV token num: {stats['oov_tokens']} ({stats['oov_ratio']:.2%})")
print(f"- In-vocab type num: {stats['vocab_size_unique_in_glove']}")



### 1.3 Cluster Word Embeddings with K-Means (Unsupervised PoS Induction)

In this step, we run **K-Means** on the (unique) word-type embeddings to induce PoS-like clusters.  
The intuition is that words with similar syntactic or semantic behaviour are close in the embedding space and therefore tend to fall into the same cluster.

#### Process
- L2-normalise all word vectors (row-wise) to stabilise distance comparisons;
- Initialise centroids with **k-means++** for better starting positions;
- Run multiple random restarts (`n_init` times), each using Lloyd’s algorithm up to `max_iter`;
- Select the run with the **lowest inertia** (tightest clusters);
- Store a **type-level** mapping `word2cluster` for downstream evaluation.

#### Inputs
- `X ∈ ℝ^{V×D}`: embedding matrix of unique in-vocab word types (same order as `vocab_words`);
- `vocab_words` (length = V): list of unique word types;
- `k`: number of clusters (here set to the number of UPOS tags);
- `n_init`, `max_iter`, `tol`: clustering hyperparameters;
- `random_state`: seed for reproducible initialisations.

#### Outputs (stored in the trained model)
- `labels_ ∈ {0,…,k−1}^V`: cluster ID for each word type;
- `centroids_ ∈ ℝ^{k×D}`: learned cluster centroids;
- `inertia_`: within-cluster sum of squared distances (lower is better);
- `n_iter_`: number of iterations used in the best run;
- `best_seed_`: random seed corresponding to the best run;
- `word2cluster: Dict[str, int]`: mapping from word type to cluster ID, later used to assign token-level labels.

Note: The `labels_` are cluster labels, not gold PoS tags. They are later used for evaluation against the gold labels (PoS tags) at the token level to compute evaluation metrics (e.g., Mutual Information, Normalised Mutual Information, V-measure, Variation of Information).


In [ ]:
import numpy as np
from typing import List, Dict, Optional

class KMeansPOSTagger:
    """
    Minimal K-Means implementation for unsupervised PoS induction.
    - Uses L2 normalisation on input embeddings.
    - Uses k-means++ for centroid initialisation.
    - Runs multiple random restarts and keeps the best result.
    - Stores word2cluster mapping for evaluation.
    """

    def __init__(
        self,
        k: int,
        n_init: int = 10,
        max_iter: int = 100,
        tol: float = 1e-4,
        random_state: Optional[int] = None,
        init: str = "k-means++",
    ):
        """
        Parameters
        ----------
        k : int
            Number of clusters (here equal to the number of UPOS tags).
        n_init : int
            Number of random restarts. The best run (lowest inertia) is kept.
        max_iter : int
            Maximum number of iterations for Lloyd’s algorithm in each run.
        tol : float
            Convergence tolerance. If centroid shift or inertia improvement is smaller than this value, stop iterating.
        random_state : Optional[int]
            Seed for reproducibility. If None, uses random seed.
        init : str
            Initialisation method for centroids. 
            Supported: "k-means++" (default) or "random".
        """
        self.k = k
        self.n_init = n_init
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.init = init

        # Learned attributes (after clustering)
        self.vocab_: List[str] = [] # list of unique in-vocab word types
        self.labels_: Optional[np.ndarray] = None # cluster label per word type
        self.centroids_: Optional[np.ndarray] = None # learned centroid matrix [k, d]
        self.inertia_: Optional[float] = None # final inertia
        self.n_iter_: Optional[int] = None # iterations in the best run
        self.best_seed_: Optional[int] = None # random seed of best run
        self.word2cluster_: Dict[str, int] = {} # mapping word -> cluster ID

    @staticmethod
    def _l2norm_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
        """
        L2-normalise each row of the embedding matrix X to unit length.
        This ensures all vectors have the same scale for clustering.
        """
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        return X / (norms + eps)

    @staticmethod
    def _kmeans_pp_init(X: np.ndarray, k: int, rng) -> np.ndarray:
        """
        Initialise centroids using k-means++:
        - Choose the first centroid randomly.
        - Choose each subsequent centroid with probability proportional to its squared distance to the nearest centroid.
        """
        n, d = X.shape
        centroids = np.empty((k, d), dtype=X.dtype)
        first = int(rng.integers(0, n))
        centroids[0] = X[first]
        closest_sq = np.full(n, np.inf, dtype=X.dtype)
        for i in range(1, k):
            dists = np.linalg.norm(X - centroids[i-1], axis=1) ** 2
            closest_sq = np.minimum(closest_sq, dists)
            probs = closest_sq / (closest_sq.sum() + 1e-12)
            idx = int(rng.choice(n, p=(probs / probs.sum())))
            centroids[i] = X[idx]
        return centroids

    @staticmethod
    def _assign(X: np.ndarray, C: np.ndarray) -> np.ndarray:
        """
        Assign each point in X to the nearest centroid in C.
        Distance metric: squared Euclidean distance.
        Returns
        -------
        labels : np.ndarray
            Shape (n,), cluster index for each data point.
        """
        x2 = (X**2).sum(axis=1, keepdims=True)
        c2 = (C**2).sum(axis=1, keepdims=True).T
        d2 = x2 + c2 - 2 * (X @ C.T)
        return d2.argmin(axis=1)

    @staticmethod
    def _update(X: np.ndarray, y: np.ndarray, k: int) -> np.ndarray:
        """
        Recompute cluster centroids:
        - For each cluster, take the mean of all assigned points.
        """
        d = X.shape[1]
        C = np.zeros((k, d), dtype=X.dtype)
        for i in range(k):
            m = (y == i)
            C[i] = X[m].mean(axis=0) if m.any() else X[np.random.randint(0, len(X))]
        return C

    @staticmethod
    def _inertia(X: np.ndarray, y: np.ndarray, C: np.ndarray) -> float:
        """
        Compute total within-cluster sum of squared distances (inertia).
        Lower inertia = tighter clusters.
        """
        D = X - C[y]
        return float((D**2).sum())

    def fit(self, X: np.ndarray, vocab_words: List[str]):
        """
        Run K-Means clustering on the embedding matrix.

        Steps:
        1. Use L2 normalisation on X.
        2. Run multiple random restarts (n_init times).
        3. For each run:
           - Initialise centroids (k-means++ or random)
           - Run Lloyd's algorithm until convergence or max_iter
           - Track best run with minimal inertia
        4. Save best clustering result and build word2cluster mapping.

        Parameters
        ----------
        X : np.ndarray
            Word embedding matrix of shape [V, D], where V is vocab size and D is embedding dim.
        vocab_words : List[str]
            List of word types corresponding to the rows in X.
        """
        assert len(vocab_words) == X.shape[0], "vocab length must match X rows"
        self.vocab_ = list(vocab_words)

        # L2 normalisation
        X_in = self._l2norm_rows(X)

        # Multiple restarts
        rng_base = np.random.default_rng(self.random_state)
        best = None

        for _ in range(self.n_init):
            seed = int(rng_base.integers(0, 2**31 - 1))
            rng = np.random.default_rng(seed)

            # Initialisation
            if self.init == "k-means++":
                C = self._kmeans_pp_init(X_in, self.k, rng)
            else:
                C = X_in[rng.choice(len(X_in), size=self.k, replace=False)].copy()

            # Iterations
            prev_inertia = None
            for it in range(self.max_iter):
                y = self._assign(X_in, C)
                C_new = self._update(X_in, y, self.k)
                inertia = self._inertia(X_in, y, C_new)
                shift = np.linalg.norm(C_new - C)
                C = C_new
                if prev_inertia is not None and abs(prev_inertia - inertia) < self.tol:
                    break
                if shift < self.tol:
                    break
                prev_inertia = inertia

            # Save run
            res = {"labels": y, "centroids": C, "inertia": inertia, "n_iter": it + 1, "seed": seed}
            if best is None or res["inertia"] < best["inertia"]:
                best = res

        # Save best result
        self.labels_ = best["labels"]
        self.centroids_ = best["centroids"]
        self.inertia_ = best["inertia"]
        self.n_iter_ = best["n_iter"]
        self.best_seed_ = best["seed"]
        self.word2cluster_ = {w: int(c) for w, c in zip(self.vocab_, self.labels_)}

        # Print logs
        print(" K-Means finished")
        print(f"Inertia: {self.inertia_:.4f}")
        print(f"Iterations: {self.n_iter_}")
        print(f"Best init seed: {self.best_seed_}")

        return self


# Run K-Means clustering
k = len(unique_upos)  # number of clusters = number of UPOS tags
tagger = KMeansPOSTagger(k=k, n_init=50, max_iter=200, random_state=42)
tagger.fit(X, vocab_words)
word2cluster = tagger.word2cluster_




### 1.4 Evaluation: Cluster Quality vs. Gold PoS Tags (20%)

In this step, we evaluate how well the induced clusters align with the gold PoS tags from the corpus.  
We build a contingency matrix between cluster assignments and gold labels and compute several widely used information-theoretic evaluation metrics.

1. Map tokens to cluster IDs:  
   - Each token is assigned to the cluster of its corresponding word type.

2. Build contingency matrix:  
   - Rows correspond to predicted clusters;  
   - Columns correspond to gold PoS tags;  
   - Each cell counts how many tokens from a given cluster belong to each gold tag.

3. Compute evaluation metrics:
   - Mutual Information (MI): Measures how much information cluster assignments and gold labels share.  
   - Normalised Mutual Information (NMI): Normalises MI by the entropy of both cluster and tag distributions.  
   - Homogeneity: Measures whether each cluster contains only members of a single gold class.  
   - Completeness: Measures whether all members of a gold class are assigned to the same cluster.  
   - V-measure: Harmonic mean of Homogeneity and Completeness.  
   - Variation of Information (VI): Distance between cluster and gold distributions (lower is better).

In [ ]:
import numpy as np
from typing import List, Tuple

def build_contingency(
    cluster_ids: List[int],
    gold_tags: List[str],
) -> Tuple[np.ndarray, List[int], List[str]]:
    """
    Build contingency matrix C[cluster, tag].
    Labels are automatically sorted internally.
    """
    # Please complete your code here.
    return C, clusters, tags

def entropy(p: np.ndarray) -> float:
    """Compute Shannon entropy H(p)."""
    # Please complete your code here.
    return float(H)

def mutual_information(C: np.ndarray) -> float:
    """Compute mutual information I(C;T)."""
    # Please complete your code here.
    return float(I)

def normalised_mutual_information(C: np.ndarray) -> float:
    """Compute normalised mutual information (NMI) with sqrt normalisation."""
    # Please complete your code here.
    return float(NMI)

def homogeneity(C: np.ndarray) -> float:
    """Compute homogeneity."""
    # Please complete your code here.
    return float(H)

def completeness(C: np.ndarray) -> float:
    """Compute completeness."""
    # Please complete your code here.
    return float(Comp)

def v_measure(C: np.ndarray) -> float:
    """Compute V-measure."""
   # Please complete your code here.
    return float(V)

def variation_of_information(C: np.ndarray) -> float:
    """Compute Variation of Information (VI)."""
    # Please complete your code here.
    return float(VI)

# Map each token in the corpus to its cluster ID from KMeans
cluster_ids_for_tokens = [word2cluster[w] for w in input_tokens]

# Evaluate the PoS tagging performance
C, clusters, tags = build_contingency(cluster_ids_for_tokens, gold_tags)
I_score = mutual_information(C)
NMI_score = normalised_mutual_information(C)
H_score = homogeneity(C)
Cmp_score = completeness(C)
V_score = v_measure(C)
VI_score = variation_of_information(C)

print(f"Mutual Information (I): {I_score:.4f}")
print(f"Normalised Mutual Info (NMI): {NMI_score:.4f}")
print(f"Homogeneity: {H_score:.4f}")
print(f"Completeness: {Cmp_score:.4f}")
print(f"V-measure: {V_score:.4f}")
print(f"Variation of Information: {VI_score:.4f}")

### 1.5 Visualisation of Clustering Results with SVD

In this step, we use SVD to project the high-dimensional word embeddings into 2D space, and visualise the clustering.

In [ ]:
!pip install scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from collections import Counter

def plot_clusters_svd(
    X: np.ndarray,
    cluster_ids: np.ndarray,
    vocab: list,
    input_tokens: list,
    topn_per_cluster: int = 20,
    figsize: tuple = (10, 8),
    random_state: int = 42
):
    """
    Plot clustered embeddings after SVD dimensionality reduction.
    Each point is a word, colored by cluster ID.
    Only top N words per cluster are plotted based on token frequencies.

    Parameters
    ----------
    X : np.ndarray [V, D]
        Word embedding matrix (same order as vocab).
    cluster_ids : np.ndarray [V]
        Cluster assignments for each word type.
    vocab : list[str]
        Vocabulary list aligned with X.
    input_tokens : list[str]
        Token-level data used to compute real frequencies.
    topn_per_cluster : int
        Number of top frequent words per cluster to show.
    figsize : tuple
        Figure size.
    random_state : int
        Seed for reproducible SVD projection.
    """
    svd = TruncatedSVD(n_components=2, random_state=random_state)
    X2 = svd.fit_transform(X)

    freq = Counter(input_tokens)

    unique_clusters = sorted(set(cluster_ids))
    selected_indices = []
    for c in unique_clusters:
        idxs = [i for i, cid in enumerate(cluster_ids) if cid == c]
        idxs_sorted = sorted(idxs, key=lambda i: freq[vocab[i]], reverse=True)[:topn_per_cluster]
        selected_indices.extend(idxs_sorted)

    selected_indices = np.array(selected_indices)

    plt.figure(figsize=figsize)
    scatter = plt.scatter(
        X2[:, 0],
        X2[:, 1],
        c=cluster_ids,
        s=10,
        cmap="tab20",
        alpha=0.5
    )

    plt.title(f"SVD visualisation of clustered embeddings (Top-{topn_per_cluster} per cluster)")
    plt.xlabel("SVD Component 1")
    plt.ylabel("SVD Component 2")

    for idx in selected_indices:
        plt.text(
            X2[idx, 0],
            X2[idx, 1],
            vocab[idx],
            fontsize=8,
            alpha=0.9,
            ha='center',
            va='center'
        )

    handles, labels = scatter.legend_elements(num=len(unique_clusters))
    plt.legend(handles, [f"C{c}" for c in unique_clusters], loc="best", title="Cluster")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_clusters_svd(
    X=X,
    cluster_ids=tagger.labels_,
    vocab=vocab_words,
    input_tokens=input_tokens,
    topn_per_cluster=5,   
    figsize=(12, 10),
    random_state=42
)

### 1.6 Analysis (20%)
Based on the completed code, please conduct experiments to analyse different versions of GloVe (https://nlp.stanford.edu/projects/glove/). You can repeat the above procedures by using different GloVe models and compare their results.

## 2. Apply Word2Vec to Unsupervised Extractive Summarisation

In this section, we use Word2Vec embeddings to perform unsupervised extractive summarisation.  
The core idea of extractive summarisation is to select the most representative and informative sentences from the original document to form a concise summary without rewriting.


In this task, we use TextRank to summarise CNNDM dataset.

### 2.1 Load Document-Summary Paris from CNNDM
We use the first 100 lines of data for experiments. Please download the test.csv file from the Moodle page. To ensure text quality and consistency, we perform text cleaning before using the data as model inputs.

In [ ]:
import pandas as pd
from typing import Tuple

import re
import html

def clean_text(s: str) -> str:
    s = html.unescape(s)
    s = s.replace("\xa0", " ").replace("\u200b", "").replace("\ufeff", "")
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    s = s.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    s = s.replace("–", "-").replace("—", "-").replace("…", "...")
    s = re.sub(r"http\S+|www\.\S+", "", s)
    s = re.sub(r"\s+", " ", s)
    s = s.strip().lower()
    return s


def extract_cnndm_test_first_n(
    csv_path: str,
    n: int = 100,
    article_col: str = "article",
    summary_col: str = "highlights",
) -> Tuple[list, list]:
    """
    Extract the first N samples from a CNNDM CSV file,
    clean & lowercase text, and return (inputs, references).
    """
    df = pd.read_csv(csv_path)
    assert article_col in df.columns and summary_col in df.columns, \
        f"Missing required columns: '{article_col}' or '{summary_col}'"

    n = min(n, len(df))
    sub = df.iloc[:n].copy()

    inputs = [clean_text(str(x)) for x in sub[article_col].fillna("")]
    references = [clean_text(str(x)) for x in sub[summary_col].fillna("")]

    return inputs, references

cnndm_path = "path_to_your_file" # Replace path_to_your_file with the path to the cnndm csv file.
docs, summaries = extract_cnndm_test_first_n(cnndm_path, n=100)


### 2.2 Convert Documents into Sentence Embeddings

In this step, we split each document into sentences and convert them into embeddings:
- Use nltk sent_tokenize for sentence segmentation.
- Tokenise each sentence and look up GloVe embeddings.
- Compute the average of token vectors to get a sentence embedding (use zero vector if no tokens found).
- Record each sentence’s document index.

Outputs inclue:
- sentence_embeddings: matrix of all sentence vectors
- sentences: corresponding sentence texts
- doc_index: mapping of each sentence to its document.

In [ ]:
import numpy as np
import nltk
from typing import List, Tuple

nltk.download('punkt')

def compute_sentence_embeddings(
    docs: List[str],
    glove_vectors: dict,
) -> Tuple[np.ndarray, List[str], List[int]]:
    """
    Segment documents into sentences and compute sentence embeddings
    by averaging pre-trained GloVe word embeddings.

    Parameters
    ----------
    docs : list[str]
        List of input article texts.
    glove_vectors : dict
        Pre-trained GloVe word embeddings {token: vector}.

    Returns
    -------
    sentence_embeddings : np.ndarray [num_sentences, dim]
        Sentence-level embedding matrix.
    sentences : list[str]
        List of sentences in the same order as embeddings.
    doc_index : list[int]
        Document index for each sentence (e.g., sentence i belongs to docs[doc_index[i]]).
    """
    all_sentences = []
    doc_index = []

    for doc_id, text in enumerate(docs):
        sents = nltk.sent_tokenize(text)
        all_sentences.extend(sents)
        doc_index.extend([doc_id] * len(sents))

    sentence_vecs = []
    for sent in all_sentences:
        tokens = sent.split()  
        vecs = [glove_vectors[w] for w in tokens if w in glove_vectors]
        if len(vecs) == 0:
            sentence_vecs.append(np.zeros(next(iter(glove_vectors.values())).shape))
        else:
            sentence_vecs.append(np.mean(vecs, axis=0))

    sentence_embeddings = np.vstack(sentence_vecs)

    return sentence_embeddings, all_sentences, doc_index


sentence_embeddings, sentences, doc_index = compute_sentence_embeddings(docs, glove_vectors)

### 2.3 TextRank for Unsupervised Summarisation. (30%)

Please refer to this paper for the algorithm https://aclanthology.org/W04-3252.pdf.

You can directly call the PageRank algorithm through `nextworkx`: https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html

In [ ]:
import numpy as np
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from typing import List

class TextRankSummarizer:
    """
    Minimal TextRank implementation for unsupervised extractive summarisation.
    Uses PageRank over a sentence similarity graph.
    """

    def __init__(self, topk: int = 3):
        """
        Parameters
        ----------
        topk : int
            Number of sentences to extract per document.
        """
        self.topk = topk

    # Please complete your code here.

In [ ]:
textrank = TextRankSummarizer(topk=3)

system_summaries = textrank.summarise(sentence_embeddings, sentences, doc_index)
print(system_summaries[0])


### 2.4 Evaluation using Rouge

We evaluate the quality of the system-generated summaries (system_summaries) against the gold references (summaries) using ROUGE metrics.
ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a standard evaluation metric for summarisation, measuring the lexical overlap between system outputs and human-written summaries.

You can install the rouge package by:
```
!pip install rouge-score
```

In [ ]:
from rouge_score import rouge_scorer

def evaluate_rouge(system_summaries, gold_summaries):
    """
    Compute average ROUGE-1/2/L F1 scores over the dataset.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
    n = len(system_summaries)

    for sys_sum, gold_sum in zip(system_summaries, gold_summaries):
        s = scorer.score(gold_sum, sys_sum)
        for key in scores:
            scores[key] += s[key].fmeasure

    for key in scores:
        scores[key] /= n
    return scores

rouge_scores = evaluate_rouge(system_summaries, summaries)
print("ROUGE scores:", rouge_scores)


### 2.5 Discussion and Analysis (30%)

Please discuss the result compared with other extractive models, such as the Lead-N model (https://arxiv.org/pdf/1908.08345). You may need to perform additional experiments to demonstrate your analysis (please write and excute your code below).